# Majority Vote: 患者単位の多数決集約によるROP分類性能評価

## 概要
Top-K品質フィルタリングで選出した画像群の予測を、患者（video_id）単位で多数決集約し、
画像単位評価と比較する。

## 集約方法
- **Hard Vote**: 予測クラスの最頻値（mode）
- **Soft Vote**: 確率の平均値 → argmax / threshold

## 比較条件
| 条件 | 画像数 | 集約方法 |
|------|--------|----------|
| All (per-image) | 6,448 | なし（ベースライン） |
| Top-10 Majority | ~10/video | Hard/Soft vote |
| Top-5 Majority | ~5/video | Hard/Soft vote |

## 評価タスク
- Multiclass: Zone (3cls), Stage (4cls), Plus (3cls)
- Binary: AROP, Treatment, RW-ROP (derived)

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from scipy import stats
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)

# Module import
sys.path.insert(0, str(Path('.').resolve()))
from select_best_images import minmax_norm

# === Paths ===
PRED_PATH = Path('outputs_clinical_v3/predictions.csv')
KUBOTA_EXCEL = Path(r'E:\Multicenter_ROP_study\Multicenter_images\Kubota_selection\selected_images_disc_retina.xlsx')
TOP_EXCEL = Path(r'E:\Multicenter_ROP_study\Multicenter_images\selected_images_disc_retina.xlsx')

# === Constants ===
EDGE_COVERAGE_CUTOFF = 0.80
WEIGHT_RETINA = 0.4
WEIGHT_GRAD = 0.4
WEIGHT_MBSS = 0.2

print('Imports done.')

Imports done.


In [2]:
# === Cell 2: Data Loading & Merge (reuse from evaluate_top10) ===

pred_df = pd.read_csv(PRED_PATH)
pred_df['image_name'] = pred_df['image_path'].apply(lambda x: Path(x).name)
print(f'predictions.csv: {len(pred_df)} images, {pred_df["video_id"].nunique()} video_ids')

# Kubota Excel features
feat_cols = ['retina_ratio', 'mbss_Grad_p90', 'mbss_score', 'disc_detected', 'disc_edge_coverage_ratio']
kubota_df = pd.read_excel(KUBOTA_EXCEL)
avail = [c for c in feat_cols if c in kubota_df.columns]
kubota_features = kubota_df[['image_name'] + avail].drop_duplicates(subset='image_name', keep='first')
merged_df = pred_df.merge(kubota_features, on='image_name', how='left')

# Fill from top-level Excel
tl = pd.read_excel(TOP_EXCEL)
tl_feat = [c for c in avail if c in tl.columns]
if 'image_name' in tl.columns and tl_feat:
    tl_features = tl[['image_name'] + tl_feat].drop_duplicates(subset='image_name', keep='first').set_index('image_name')
    mm = merged_df['retina_ratio'].isna()
    for col in tl_feat:
        fill = merged_df.loc[mm, 'image_name'].map(tl_features[col])
        merged_df.loc[mm, col] = fill.values

n_has_feat = merged_df['retina_ratio'].notna().sum()
print(f'Features available: {n_has_feat}/{len(merged_df)} ({n_has_feat/len(merged_df)*100:.1f}%)')

# Verify label consistency per video_id
label_cols = ['zone_label', 'stage_label', 'plus_label', 'aggressive_rop_label', 'treatment_label']
inconsistent = []
for vid, group in merged_df.groupby('video_id'):
    for col in label_cols:
        if group[col].nunique() > 1:
            inconsistent.append((vid, col, group[col].unique()))
if inconsistent:
    print(f'WARNING: {len(inconsistent)} label inconsistencies found!')
    for vid, col, vals in inconsistent[:5]:
        print(f'  {vid}: {col} = {vals}')
else:
    print('Label consistency check: OK (all labels consistent within video_id)')

predictions.csv: 6448 images, 347 video_ids
Features available: 6448/6448 (100.0%)
Label consistency check: OK (all labels consistent within video_id)


In [3]:
# === Cell 3: Select Top-10 and Top-5 ===

def select_top_k_per_video(df, top_k=10, edge_cov_cutoff=0.80):
    """C-rule3_thr0.80 per video_id"""
    all_selected = []
    for vid, group in df.groupby('video_id'):
        valid = group[group['retina_ratio'].notna() & (group['retina_ratio'] > 0)].copy()
        if len(valid) == 0:
            continue
        stage1 = valid[
            (valid['disc_detected'] == True) &
            valid['disc_edge_coverage_ratio'].notna() &
            (valid['disc_edge_coverage_ratio'] >= edge_cov_cutoff)
        ].copy()
        if len(stage1) > 0:
            stage1['retina_norm'] = minmax_norm(stage1['retina_ratio'].fillna(0))
            stage1['grad_norm'] = minmax_norm(stage1['mbss_Grad_p90'].fillna(0))
            stage1['mbss_norm'] = minmax_norm(stage1['mbss_score'].fillna(0))
            stage1['quality_score'] = (
                WEIGHT_RETINA * stage1['retina_norm'] +
                WEIGHT_GRAD * stage1['grad_norm'] +
                WEIGHT_MBSS * stage1['mbss_norm']
            )
            stage1 = stage1.sort_values('quality_score', ascending=False)
            sel = stage1.head(top_k).copy()
        else:
            sel = pd.DataFrame()
        nr = top_k - len(sel)
        if nr > 0:
            rem = valid[~valid.index.isin(sel.index)].sort_values('retina_ratio', ascending=False)
            sel = pd.concat([sel, rem.head(nr)])
        if len(sel) > 0:
            all_selected.append(sel)
    return pd.concat(all_selected, ignore_index=True) if all_selected else pd.DataFrame()

top10_df = select_top_k_per_video(merged_df, top_k=10)
top5_df = select_top_k_per_video(merged_df, top_k=5)

print(f'All:    {len(merged_df):>5} images, {merged_df["video_id"].nunique()} video_ids')
print(f'Top-10: {len(top10_df):>5} images, {top10_df["video_id"].nunique()} video_ids')
print(f'Top-5:  {len(top5_df):>5} images, {top5_df["video_id"].nunique()} video_ids')

All:     6448 images, 347 video_ids
Top-10:  3089 images, 347 video_ids
Top-5:   1650 images, 347 video_ids


In [4]:
# === Cell 4: Majority Vote Aggregation Functions ===

def aggregate_hard_vote(group, pred_col):
    """Hard majority vote: mode of predictions"""
    mode_result = stats.mode(group[pred_col], keepdims=True)
    return int(mode_result.mode[0])


def aggregate_soft_vote_multiclass(group, prob_cols):
    """Soft vote: mean probabilities -> argmax"""
    mean_probs = group[prob_cols].mean()
    return int(np.argmax(mean_probs.values))


def aggregate_soft_vote_binary(group, prob_col):
    """Soft vote for binary: mean probability"""
    return float(group[prob_col].mean())


def aggregate_per_video(df, method='soft'):
    """
    video_id単位で予測を集約
    
    method: 'hard' (mode) or 'soft' (mean probability)
    Returns: DataFrame with one row per video_id
    """
    rows = []
    for vid, group in df.groupby('video_id'):
        row = {'video_id': vid, 'n_images': len(group)}
        
        # Ground truth (same for all images in video)
        row['zone_label'] = int(group['zone_label'].iloc[0])
        row['stage_label'] = int(group['stage_label'].iloc[0])
        row['plus_label'] = int(group['plus_label'].iloc[0])
        row['aggressive_rop_label'] = int(group['aggressive_rop_label'].iloc[0])
        row['treatment_label'] = int(group['treatment_label'].iloc[0])
        
        if method == 'hard':
            # Hard vote: mode of predictions
            row['zone_pred'] = aggregate_hard_vote(group, 'zone_pred')
            row['stage_pred'] = aggregate_hard_vote(group, 'stage_pred')
            row['plus_pred'] = aggregate_hard_vote(group, 'plus_pred')
            row['aggressive_rop_pred'] = aggregate_hard_vote(group, 'aggressive_rop_pred')
            row['treatment_pred'] = aggregate_hard_vote(group, 'treatment_pred')
            
            # For binary AUC calculation, still need probabilities
            row['aggressive_rop_prob_1'] = float(group['aggressive_rop_prob_1'].mean())
            row['treatment_prob_1'] = float(group['treatment_prob_1'].mean())
            row['zone_prob_0'] = float(group['zone_prob_0'].mean())
            row['stage_prob_3'] = float(group['stage_prob_3'].mean())
            row['plus_prob_2'] = float(group['plus_prob_2'].mean())
            
        else:  # soft
            # Soft vote: mean probabilities -> argmax / threshold
            zone_probs = group[['zone_prob_0', 'zone_prob_1', 'zone_prob_2']].mean()
            row['zone_pred'] = int(np.argmax(zone_probs.values))
            row['zone_prob_0'] = float(zone_probs['zone_prob_0'])
            
            stage_probs = group[['stage_prob_0', 'stage_prob_1', 'stage_prob_2', 'stage_prob_3']].mean()
            row['stage_pred'] = int(np.argmax(stage_probs.values))
            row['stage_prob_3'] = float(stage_probs['stage_prob_3'])
            
            plus_probs = group[['plus_prob_0', 'plus_prob_1', 'plus_prob_2']].mean()
            row['plus_pred'] = int(np.argmax(plus_probs.values))
            row['plus_prob_2'] = float(plus_probs['plus_prob_2'])
            
            arop_prob = float(group['aggressive_rop_prob_1'].mean())
            row['aggressive_rop_prob_1'] = arop_prob
            row['aggressive_rop_pred'] = int(arop_prob >= 0.5)
            
            treat_prob = float(group['treatment_prob_1'].mean())
            row['treatment_prob_1'] = treat_prob
            row['treatment_pred'] = int(treat_prob >= 0.5)
        
        rows.append(row)
    
    return pd.DataFrame(rows)

print('Aggregation functions defined.')

Aggregation functions defined.


In [5]:
# === Cell 5: Evaluation Functions ===

def compute_multiclass_metrics(y_true, y_pred):
    """Zone/Stage/Plus: accuracy, kappa, f1_macro"""
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'kappa': cohen_kappa_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }


def compute_binary_metrics(y_true, y_pred, y_prob=None):
    """AROP/Treatment/RW-ROP: sensitivity, specificity, PPV, NPV, F1, AUC"""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    result = {
        'sensitivity': sens, 'specificity': spec,
        'PPV': ppv, 'NPV': npv, 'F1': f1,
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
    }
    if y_prob is not None and len(set(y_true)) >= 2:
        result['AUC'] = roc_auc_score(y_true, y_prob)
    return result


def compute_rw_rop(df):
    """RW-ROP: hard prediction (OR) + soft probability"""
    rw_true = ((df['plus_label'] == 2) | (df['stage_label'] == 3) | (df['zone_label'] == 0)).astype(int)
    rw_pred = ((df['plus_pred'] == 2) | (df['stage_pred'] == 3) | (df['zone_pred'] == 0)).astype(int)
    rw_prob = 1 - ((1 - df['plus_prob_2']) * (1 - df['stage_prob_3']) * (1 - df['zone_prob_0']))
    return rw_true, rw_pred, rw_prob


def evaluate_all_tasks(df):
    """全タスク一括評価"""
    results = {}
    results['zone'] = compute_multiclass_metrics(df['zone_label'], df['zone_pred'])
    results['stage'] = compute_multiclass_metrics(df['stage_label'], df['stage_pred'])
    results['plus'] = compute_multiclass_metrics(df['plus_label'], df['plus_pred'])
    results['aggressive_rop'] = compute_binary_metrics(
        df['aggressive_rop_label'], df['aggressive_rop_pred'], df['aggressive_rop_prob_1'])
    results['treatment'] = compute_binary_metrics(
        df['treatment_label'], df['treatment_pred'], df['treatment_prob_1'])
    rw_true, rw_pred, rw_prob = compute_rw_rop(df)
    results['rw_rop'] = compute_binary_metrics(rw_true, rw_pred, rw_prob)
    return results

print('Evaluation functions defined.')

Evaluation functions defined.


In [6]:
# === Cell 6: Aggregate & Evaluate ===

# Per-image baseline (all images)
print('=== Per-Image Baseline (All 6,448 images) ===')
results_all_img = evaluate_all_tasks(merged_df)

# Majority vote aggregations
conditions = {}

for method in ['hard', 'soft']:
    for label, src_df in [('All', merged_df), ('Top-10', top10_df), ('Top-5', top5_df)]:
        key = f'{label}_{method}'
        agg_df = aggregate_per_video(src_df, method=method)
        conditions[key] = {
            'agg_df': agg_df,
            'results': evaluate_all_tasks(agg_df),
            'n_videos': len(agg_df),
        }
        print(f'{key}: {len(agg_df)} videos aggregated')

print(f'\nTotal conditions: {len(conditions)}')

=== Per-Image Baseline (All 6,448 images) ===
All_hard: 347 videos aggregated
Top-10_hard: 347 videos aggregated
Top-5_hard: 347 videos aggregated
All_soft: 347 videos aggregated
Top-10_soft: 347 videos aggregated
Top-5_soft: 347 videos aggregated

Total conditions: 6


In [7]:
# === Cell 7: Comparison Table ===

# Multiclass tasks
print('='*90)
print('Multiclass Tasks (Patient-level Majority Vote)')
print('='*90)
print(f'{"Task":<8} {"Metric":<10} {"PerImg(All)":>12} {"All_hard":>10} {"All_soft":>10} '
      f'{"T10_hard":>10} {"T10_soft":>10} {"T5_hard":>10} {"T5_soft":>10}')
print('-'*90)

for task in ['zone', 'stage', 'plus']:
    for metric in ['accuracy', 'kappa', 'f1_macro']:
        base = results_all_img[task][metric]
        vals = [conditions[k]['results'][task][metric] 
                for k in ['All_hard', 'All_soft', 'Top-10_hard', 'Top-10_soft', 'Top-5_hard', 'Top-5_soft']]
        print(f'{task:<8} {metric:<10} {base:>12.4f} ' + ' '.join(f'{v:>10.4f}' for v in vals))
    print()

# Binary tasks
print('\n' + '='*90)
print('Binary Tasks (Patient-level Majority Vote)')
print('='*90)
print(f'{"Task":<15} {"Metric":<12} {"PerImg(All)":>12} {"All_hard":>10} {"All_soft":>10} '
      f'{"T10_hard":>10} {"T10_soft":>10} {"T5_hard":>10} {"T5_soft":>10}')
print('-'*90)

for task in ['treatment', 'aggressive_rop', 'rw_rop']:
    for metric in ['sensitivity', 'specificity', 'F1', 'AUC']:
        if metric not in results_all_img[task]:
            continue
        base = results_all_img[task][metric]
        vals = []
        for k in ['All_hard', 'All_soft', 'Top-10_hard', 'Top-10_soft', 'Top-5_hard', 'Top-5_soft']:
            v = conditions[k]['results'][task].get(metric, float('nan'))
            vals.append(v)
        print(f'{task:<15} {metric:<12} {base:>12.4f} ' + ' '.join(f'{v:>10.4f}' for v in vals))
    print()

Multiclass Tasks (Patient-level Majority Vote)
Task     Metric      PerImg(All)   All_hard   All_soft   T10_hard   T10_soft    T5_hard    T5_soft
------------------------------------------------------------------------------------------
zone     accuracy         0.7323     0.7522     0.7666     0.7550     0.7752     0.7464     0.7579
zone     kappa            0.5501     0.5801     0.6040     0.5823     0.6161     0.5713     0.5895
zone     f1_macro         0.7132     0.7354     0.7469     0.7358     0.7556     0.7276     0.7419

stage    accuracy         0.7298     0.7378     0.7320     0.7320     0.7233     0.7262     0.7176
stage    kappa            0.6224     0.6343     0.6270     0.6266     0.6144     0.6185     0.6070
stage    f1_macro         0.5780     0.5789     0.5747     0.5755     0.5689     0.5723     0.5644

plus     accuracy         0.8826     0.8732     0.8847     0.8876     0.8818     0.8761     0.8847
plus     kappa            0.6433     0.6116     0.6497     0.6419   

In [8]:
# === Cell 8: Key Findings Summary ===

print('='*70)
print('Summary: Soft Vote (Top-10 vs Top-5) vs Per-Image Baseline')
print('='*70)

# Focus on soft vote (generally better for probability-based models)
summary_rows = []
for task in ['zone', 'stage', 'plus']:
    for metric in ['accuracy', 'kappa']:
        base = results_all_img[task][metric]
        t10 = conditions['Top-10_soft']['results'][task][metric]
        t5 = conditions['Top-5_soft']['results'][task][metric]
        summary_rows.append({
            'Task': task.capitalize(), 'Metric': metric,
            'PerImg_All': base,
            'MajVote_Top10': t10, 'Delta_T10': t10 - base,
            'MajVote_Top5': t5, 'Delta_T5': t5 - base,
        })

for task in ['treatment', 'aggressive_rop', 'rw_rop']:
    for metric in ['sensitivity', 'specificity', 'AUC']:
        if metric not in results_all_img[task]:
            continue
        base = results_all_img[task][metric]
        t10 = conditions['Top-10_soft']['results'][task].get(metric, float('nan'))
        t5 = conditions['Top-5_soft']['results'][task].get(metric, float('nan'))
        summary_rows.append({
            'Task': task, 'Metric': metric,
            'PerImg_All': base,
            'MajVote_Top10': t10, 'Delta_T10': t10 - base,
            'MajVote_Top5': t5, 'Delta_T5': t5 - base,
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False, float_format='{:.4f}'.format))

# Highlight improvements
print('\n--- Notable Changes (Soft Vote Top-5 vs Per-Image All) ---')
for _, row in summary_df.iterrows():
    d = row['Delta_T5']
    if abs(d) >= 0.02:
        direction = 'UP' if d > 0 else 'DOWN'
        print(f'  {row["Task"]:<15} {row["Metric"]:<12}: {row["PerImg_All"]:.4f} -> {row["MajVote_Top5"]:.4f} ({d:+.4f}) [{direction}]')

Summary: Soft Vote (Top-10 vs Top-5) vs Per-Image Baseline
          Task      Metric  PerImg_All  MajVote_Top10  Delta_T10  MajVote_Top5  Delta_T5
          Zone    accuracy      0.7323         0.7752     0.0429        0.7579    0.0256
          Zone       kappa      0.5501         0.6161     0.0659        0.5895    0.0394
         Stage    accuracy      0.7298         0.7233    -0.0065        0.7176   -0.0123
         Stage       kappa      0.6224         0.6144    -0.0080        0.6070   -0.0155
          Plus    accuracy      0.8826         0.8818    -0.0008        0.8847    0.0021
          Plus       kappa      0.6433         0.6390    -0.0043        0.6479    0.0046
     treatment sensitivity      0.9171         1.0000     0.0829        1.0000    0.0829
     treatment specificity      0.9526         0.9459    -0.0067        0.9459   -0.0067
     treatment         AUC      0.9780         0.9910     0.0130        0.9893    0.0113
aggressive_rop sensitivity      0.8596         0.88

In [9]:
# === Cell 9: RW-ROP Threshold Optimization for Majority Vote ===

print('=== RW-ROP Threshold Optimization (Soft Vote) ===')
print(f'{"Condition":<20} {"Strategy":<12} {"Threshold":>10} {"Sensitivity":>12} {"Specificity":>12}')
print('-'*70)

for label in ['All_soft', 'Top-10_soft', 'Top-5_soft']:
    agg_df = conditions[label]['agg_df']
    rw_true, _, rw_prob = compute_rw_rop(agg_df)
    
    if len(set(rw_true)) < 2:
        print(f'{label:<20} (skipped - single class)')
        continue
    
    # Default threshold
    rw_pred_05 = (rw_prob >= 0.5).astype(int)
    cm = confusion_matrix(rw_true, rw_pred_05, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens_05 = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec_05 = tn / (tn + fp) if (tn + fp) > 0 else 0
    print(f'{label:<20} {"default":<12} {0.5:>10.4f} {sens_05:>12.4f} {spec_05:>12.4f}')
    
    # Youden's J
    fpr, tpr, thresholds = roc_curve(rw_true, rw_prob)
    youden_j = tpr - fpr
    youden_idx = np.argmax(youden_j)
    thresh_y = float(thresholds[youden_idx])
    rw_pred_y = (rw_prob >= thresh_y).astype(int)
    cm_y = confusion_matrix(rw_true, rw_pred_y, labels=[0, 1])
    tn_y, fp_y, fn_y, tp_y = cm_y.ravel()
    sens_y = tp_y / (tp_y + fn_y) if (tp_y + fn_y) > 0 else 0
    spec_y = tn_y / (tn_y + fp_y) if (tn_y + fp_y) > 0 else 0
    print(f'{label:<20} {"Youden":<12} {thresh_y:>10.4f} {sens_y:>12.4f} {spec_y:>12.4f}')
    
    # Sensitivity >= 95%
    valid_idx = np.where(tpr >= 0.95)[0]
    if len(valid_idx) > 0:
        best_idx = valid_idx[np.argmin(fpr[valid_idx])]
        thresh_s = float(thresholds[best_idx])
        rw_pred_s = (rw_prob >= thresh_s).astype(int)
        cm_s = confusion_matrix(rw_true, rw_pred_s, labels=[0, 1])
        tn_s, fp_s, fn_s, tp_s = cm_s.ravel()
        sens_s = tp_s / (tp_s + fn_s) if (tp_s + fn_s) > 0 else 0
        spec_s = tn_s / (tn_s + fp_s) if (tn_s + fp_s) > 0 else 0
        print(f'{label:<20} {"sens>=95%":<12} {thresh_s:>10.4f} {sens_s:>12.4f} {spec_s:>12.4f}')
    print()

# Same for Treatment
print('\n=== Treatment Threshold Optimization (Soft Vote) ===')
print(f'{"Condition":<20} {"Strategy":<12} {"Threshold":>10} {"Sensitivity":>12} {"Specificity":>12}')
print('-'*70)

for label in ['All_soft', 'Top-10_soft', 'Top-5_soft']:
    agg_df = conditions[label]['agg_df']
    y_true = agg_df['treatment_label']
    y_prob = agg_df['treatment_prob_1']
    
    if len(set(y_true)) < 2:
        continue
    
    y_pred_05 = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred_05, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens_05 = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec_05 = tn / (tn + fp) if (tn + fp) > 0 else 0
    print(f'{label:<20} {"default":<12} {0.5:>10.4f} {sens_05:>12.4f} {spec_05:>12.4f}')
    
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    youden_j = tpr - fpr
    youden_idx = np.argmax(youden_j)
    thresh_y = float(thresholds[youden_idx])
    y_pred_y = (y_prob >= thresh_y).astype(int)
    cm_y = confusion_matrix(y_true, y_pred_y, labels=[0, 1])
    tn_y, fp_y, fn_y, tp_y = cm_y.ravel()
    sens_y = tp_y / (tp_y + fn_y) if (tp_y + fn_y) > 0 else 0
    spec_y = tn_y / (tn_y + fp_y) if (tn_y + fp_y) > 0 else 0
    print(f'{label:<20} {"Youden":<12} {thresh_y:>10.4f} {sens_y:>12.4f} {spec_y:>12.4f}')
    print()

=== RW-ROP Threshold Optimization (Soft Vote) ===
Condition            Strategy      Threshold  Sensitivity  Specificity
----------------------------------------------------------------------
All_soft             default          0.5000       0.9167       0.8251
All_soft             Youden           0.5646       0.8929       0.9049
All_soft             sens>=95%        0.4583       0.9524       0.7605

Top-10_soft          default          0.5000       0.9167       0.8403
Top-10_soft          Youden           0.6216       0.8333       0.9468
Top-10_soft          sens>=95%        0.4583       0.9524       0.7529

Top-5_soft           default          0.5000       0.9167       0.8403
Top-5_soft           Youden           0.5078       0.9167       0.8479
Top-5_soft           sens>=95%        0.4583       0.9524       0.7567


=== Treatment Threshold Optimization (Soft Vote) ===
Condition            Strategy      Threshold  Sensitivity  Specificity
-----------------------------------------

In [10]:
# === Cell 10: Save Results ===
import json

output_dir = Path('outputs_clinical_v3')

# Save all results to JSON
save_data = {
    'per_image_baseline': {task: {k: float(v) if isinstance(v, (np.floating, float)) else int(v) 
                                   for k, v in metrics.items()}
                           for task, metrics in results_all_img.items()},
}
for key, cond in conditions.items():
    save_data[key] = {
        'n_videos': cond['n_videos'],
        'results': {task: {k: float(v) if isinstance(v, (np.floating, float)) else int(v)
                           for k, v in metrics.items()}
                    for task, metrics in cond['results'].items()}
    }

json_path = output_dir / 'majority_vote_results.json'
with open(json_path, 'w') as f:
    json.dump(save_data, f, indent=2)
print(f'Results saved to {json_path}')

# Save summary table
csv_path = output_dir / 'majority_vote_summary.csv'
summary_df.to_csv(csv_path, index=False)
print(f'Summary saved to {csv_path}')

Results saved to outputs_clinical_v3\majority_vote_results.json
Summary saved to outputs_clinical_v3\majority_vote_summary.csv
